# Business Semantics

This notebook defines the shared metrics and semantic layer for the ad tech demo.

## Create the semantic views

This notebook creates reusable metric views that define the business semantics for the ad tech platform.

In [0]:
%sql
CREATE OR REPLACE VIEW adtech_demo.adtech_platform.vw_campaign_metrics AS
SELECT
  c.campaign_id,
  c.campaign_name,
  c.advertiser,
  SUM(i.viewability::INT) AS total_viewable_impressions,
  COUNT(i.impression_id) AS total_impressions,
  SUM(CASE WHEN c2.click_id IS NOT NULL THEN 1 ELSE 0 END) AS total_clicks,
  SUM(c2.cost) AS total_spend,
  SUM(co.revenue) AS total_revenue,
  AVG(i.view_duration_seconds) AS avg_view_duration_seconds,
  SUM(CASE WHEN i.view_duration_seconds >= 1 THEN 1 ELSE 0 END) / NULLIF(COUNT(i.impression_id), 0) AS viewability_rate,
  SUM(CASE WHEN c2.click_id IS NOT NULL THEN 1 ELSE 0 END) / NULLIF(COUNT(i.impression_id), 0) AS click_through_rate,
  SUM(co.revenue) / NULLIF(SUM(c2.cost), 0) AS roas,
  SUM(CASE WHEN i.view_duration_seconds >= 5 THEN 1 ELSE 0 END) / NULLIF(COUNT(i.impression_id), 0) AS engaged_view_rate
FROM adtech_demo.adtech_platform.campaigns c
LEFT JOIN adtech_demo.adtech_platform.impressions i ON c.campaign_id = i.campaign_id
LEFT JOIN adtech_demo.adtech_platform.clicks c2 ON i.impression_id = c2.impression_id
LEFT JOIN adtech_demo.adtech_platform.conversions co ON c2.click_id = co.click_id
GROUP BY c.campaign_id, c.campaign_name, c.advertiser;

In [0]:
%sql
SELECT * FROM adtech_demo.adtech_platform.vw_campaign_metrics LIMIT 10;

## Unity Catalog Metric View

Unlike the regular view above, `vw_audience_engagement` is defined as a **UC Metric View**. Key differences:

| | Regular View | Metric View |
| --- | --- | --- |
| Aggregation | Locked at creation time | Flexible at query time via `MEASURE()` |
| Dimensions | Implicit (GROUP BY columns) | Explicitly declared with metadata |
| Measures | Pre-computed in SELECT | Defined as reusable expressions |
| Semantic metadata | None | Comments, display names, synonyms, formats |
| AI/BI integration | Basic | Enhanced — Genie and dashboards understand the semantics |

In [0]:
%sql
CREATE OR REPLACE VIEW adtech_demo.adtech_platform.vw_audience_engagement
WITH METRICS
LANGUAGE YAML
AS $$
  version: 1.1
  source: >
    SELECT impression_id, audience_segment, viewability, view_duration_seconds
    FROM adtech_demo.adtech_platform.impressions
  comment: Audience engagement metrics from the ad tech impressions data.
  dimensions:
    - name: audience_segment
      expr: audience_segment
      display_name: Audience Segment
      comment: The audience segment targeted by the impression
      synonyms:
        - segment
        - audience
  measures:
    - name: Impressions
      expr: COUNT(DISTINCT impression_id)
      display_name: Impressions
      comment: Total distinct impressions served
      format:
        type: number
        decimal_places:
          type: exact
          places: 0
    - name: Viewable Impressions
      expr: SUM(CASE WHEN viewability THEN 1 ELSE 0 END)
      display_name: Viewable Impressions
      comment: Impressions that met viewability criteria
      format:
        type: number
        decimal_places:
          type: exact
          places: 0
    - name: Avg View Duration
      expr: AVG(view_duration_seconds)
      display_name: Avg View Duration (s)
      comment: Average time users spent viewing the ad in seconds
      format:
        type: number
        decimal_places:
          type: exact
          places: 2
      synonyms:
        - average watch duration
        - view time
    - name: Engaged View Rate
      expr: SUM(CASE WHEN view_duration_seconds >= 5 THEN 1 ELSE 0 END) / NULLIF(COUNT(impression_id), 0)
      display_name: Engaged View Rate
      comment: Proportion of impressions with view duration >= 5 seconds
      format:
        type: percentage
        decimal_places:
          type: exact
          places: 2
      synonyms:
        - engagement rate
$$

In [0]:
%sql
-- Metric views require MEASURE() to evaluate aggregate measures at query time
SELECT
  `audience_segment`,
  MEASURE(`Impressions`) AS impressions,
  MEASURE(`Viewable Impressions`) AS viewable_impressions,
  MEASURE(`Avg View Duration`) AS avg_view_duration_seconds,
  MEASURE(`Engaged View Rate`) AS engaged_view_rate
FROM adtech_demo.adtech_platform.vw_audience_engagement
GROUP BY `audience_segment`

## Key metrics
- click_through_rate (CTR)
- return_on_ad_spend (ROAS)
- viewability_rate
- average_watch_duration
- engaged_view_rate